# Getting started with `da6idb`

Reading and processing the files from APS 6-ID-B, with no viewer involved.

This runs against `tise2_2_00152`, the scan that travels with the repository:
a 101-point `psic_eta` rocking curve with its detector stack. Point
`FOLDER` somewhere else and the rest should follow.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# This notebook lives in examples/, so da6idb is one level up. Installing the
# package (pip install -e . from the repository root) makes this unnecessary,
# but the notebook should run either way.
REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
try:
    import da6idb
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO))
    import da6idb

FOLDER = REPO                # the repository root, which holds the example scan
SCAN = 152

## What is in the folder\n\n`open_folder` lists the scans; `summary()` is a DataFrame.

In [ ]:
folder = da6idb.open_folder(FOLDER)
folder.summary().head(10)

## One scan\n\nIndexing by scan number gives a `Scan`.

In [ ]:
scan = folder[SCAN]
scan

In [ ]:
scan.hkl_position()     # where the diffractometer was, averaged over the scan

## Curves

`curve` takes an x field and any number of y fields, and normalises if asked.
Left to itself it uses the motor the scan moved -- or the point index, for a
plan that moved nothing.

In [ ]:
curve = scan.curve(y="lambda250k_stats1_total", normalise="Ion_Chamber_1")
ax = curve.plot(marker="o", ms=3, lw=1, figsize=(7, 3.5))
ax.set_ylabel("counts / monitor"); ax.grid(alpha=0.3)

In [ ]:
scan.statistics("lambda250k_stats1_total")   # max, min, half-max centre and FWHM

## The detector frames

`images()` finds the detector file the way the viewers do, and brings the
ROIs the beamline recorded -- from the image file, or from the master's
baseline when the image has none.

In [ ]:
images = scan.images()
print(images)
images.rois

In [ ]:
# every metric for ROI 1, frame by frame -- the same arithmetic the viewer runs
images.roi_series(0).head()

In [ ]:
ax = images.roi_curve(0, "sum", background=2).plot(figsize=(7, 3), lw=1)
ax.set_ylabel("ROI1 sum, background subtracted"); ax.grid(alpha=0.3)

## Reciprocal space

`hkl()` gathers the geometry from the master, the image file and any
`hkl_setup.json` in the folder, and says where each piece came from.

In [ ]:
m = scan.hkl()
m.sources()

In [ ]:
m.resolution()    # the cell the sampling supports, and what that was measured from

### A gridded map

`grid()` bins the frames onto a regular lattice and hands back an
`xarray.DataArray` with real H, K and L coordinates. It is cached as the same
`.vti` the viewers read, so a grid made here opens there.

`oversample` scales the automatic cell size; 4 keeps this example quick.

In [ ]:
cube = m.grid(oversample=4.0)
cube

In [ ]:
# select by coordinate rather than by index, and let xarray draw it
plane = cube.sel(L=float(cube.L[len(cube.L) // 2]), method="nearest")
plane.plot(figsize=(6, 4.5), robust=True)

In [ ]:
# or sum along an axis
cube.sum("L", skipna=True).plot(figsize=(6, 4.5), robust=True)

### Slices and cuts

`slice` cuts a plane at any angle, averaging over a slab of cells if asked.
Where an image axis runs along H, K or L it is labelled with it.

In [ ]:
sl = m.slice(normal=(0, 0, 1), offset=0.0, layers=3)
print(sl.dims, sl.shape)
sl.plot(figsize=(6, 4.5), robust=True)

In [ ]:
# a line through the slice is a one-liner once it is labelled
cut = sl.sel({sl.dims[0]: sl[sl.dims[0]][len(sl[sl.dims[0]]) // 2]}, method="nearest")
ax = cut.plot(figsize=(7, 3), lw=1); import matplotlib.pyplot as plt; plt.grid(alpha=0.3)

### Writing it out\n\nText, CSV, `.npz` or HDF5, each with a header saying where it came from.

In [ ]:
from da6idb import export

out = export.write_dataarray(sl, "slice.txt")
print(out, "\n")
print("\n".join(open(out).read().splitlines()[:6]))

## A run of scans

`grid_scans` measures the cell size once, from the first scan, and reuses it
for the rest -- so with a common HKL range every grid shares origin, spacing
and shape, and they can be compared cell for cell.

In [ ]:
from da6idb import batch

print(batch.expand_scans(152, 160, 2))
# batch.grid_scans(FOLDER, [152, 153], hkl_range=(0.95, 1.10, -0.60, -0.40, 7.40, 7.65))